# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead |Achmad Rizky Akbar| |
| Member | Kajetan Zduńczyk| |
| Member *(optional)* | | |


## 2. Mission Title & Research Question

**Title:** *CEO Turnover and Executive Pay Dispersion in Europe: Evidence from Leadership Changes*

**Research question:** *Does CEO turnover causally change CEO compensation levels and pay dispersion within European firms?*

**Why it matters:** *By focusing on leadership changes observed in BoardEx, we can estimate how compensation responds to a governance shock without hand‑collecting policy data. This helps investors and regulators understand whether turnover acts as a disciplining mechanism on executive pay and internal pay gaps.*

## 3. Data

**Source(s):**  

1. **BoardEx - Individual Profile Employment**

    Includes data about Individual Profile Employment for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-employment/

2. **BoardEx - Individual Profile Details**

    Includes data about Individual Profile Details for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-details/

3. **Company Profile Details - BoardEx (Wharton Data Research Services)**

    Company Profile Details includes data such as location, market cap, and sector.
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/company-profile/company-profile-details/

4. **Fundamentals Annual - Compustat Global (Wharton Data Research Services)**

    Provides fundamental annual company information
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/global-daily/fundamentals-annual/

5. **Annual Remuneration - BoardEx (Wharton Data Research Services):**
    
    Data such as salary, bonus, and other cash compensation
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/compensation-analysis/annual-remuneration/

6. **Firms in Stoxx 600 Index**

    STOXX 600 is a major stock index representing the performance of 600 large-, mid-, and small-capitalization companies across 17 developed European countries.

    https://www.stoxx.com/selection-lists

**Unit of observation:** *Executive–year within a firm (linked to firm-year fundamentals and leadership changes).*

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
| Total CEO compensation | Numeric | **Target** | Total annual compensation (salary + bonus + other cash/stock, in EUR) |
| Pay dispersion | Numeric | **Target / Outcome** | CEO pay relative to top-executive team (e.g., CEO-to-top-5 ratio) |
| CEO turnover indicator | Binary | **Treatment** | Flag for CEO change in a given year (from BoardEx role start/end) |
| Post‑turnover period | Binary | Feature | Indicator for years after turnover (event window) |
| Firm size (log assets / market cap) | Numeric | Feature | Scale and visibility of firm |
| Profitability (ROA / EBIT margin) | Numeric | Feature | Performance controls |
| Leverage | Numeric | Feature | Capital structure |
| Industry & country fixed effects | Categorical | Feature | Sector and institutional context |
| Executive tenure | Numeric | Feature | Human capital and bargaining power |

**Potential data quality issues:**  
- **Missing compensation components:** use multiple imputation or restrict to firms with complete pay breakdowns; report sensitivity to this choice.
- **Turnover date ambiguity:** define CEO change using role start/end dates and validate with overlapping roles; conduct robustness with alternative windows.
- **Reporting bias / top-coding:** winsorize extreme pay values; compare distributions by country to detect systematic reporting differences.
- **Selection bias in BoardEx coverage:** include a Stoxx 600 filter and check representativeness vs. population benchmarks.
- **Timing misalignment:** align fiscal-year fundamentals with compensation year; drop or lag inconsistent observations.
- **Currency and inflation effects:** convert to EUR and deflate using CPI to ensure comparability across years.

---

In [187]:
import numpy as np
import pandas as pd
import wrds
from pathlib import Path

In [188]:
DATA_DIR = Path("..") / ".." / "data" / "data_project"

In [189]:
db = wrds.Connection()

# Verify available BoardEx tables (run once to confirm table names)
# boardex_tables = db.list_tables("boardex")
# print([t for t in boardex_tables if any(k in t.lower() for k in ("company","remuner","employ","detail"))])

def load_or_fetch(cache_path, query_fn, label=""):
    """Load from local CSV cache; query WRDS and cache on first run."""
    if cache_path.exists():
        df = pd.read_csv(cache_path)
        print(f"[cache] {label}: {len(df):,} rows")
    else:
        print(f"[WRDS]  {label}: querying...")
        df = query_fn()
        df.to_csv(cache_path, index=False)
        print(f"[WRDS]  {label}: {len(df):,} rows — cached to {cache_path.name}")
    return df

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [190]:
# STOXX 600 stays as CSV (external index list, not a WRDS table)
df_sxxp = pd.read_csv(DATA_DIR / "stoxx600_clean.csv", sep=";")
stoxx_isins   = tuple(df_sxxp["ISIN"].tolist())
stoxx_isins_s = str(stoxx_isins)   # used inside SQL IN clauses
df_sxxp.head()

,Creation_Date,Internal_Key,ISIN,RIC,Instrument_Name,Country,Currency,Exchange,Index Membership,Rank (FINAL)
0,20260501,546078,NL0010273215,ASML.AS,ASML HLDG,NL,EUR,Euronext Amsterdam,Large,1
1,20260501,40054,GB0005405286,HSBA.L,HSBC,GB,GBP,London SE,Large,2
2,20260501,98952,GB0009895292,AZN.L,ASTRAZENECA,GB,GBP,London SE,Large,3
3,20260501,474577,CH0012032048,ROPC.S,ROCHE PS,CH,CHF,Six Swiss Exchange,Large,4
4,20260501,477408,CH0012005267,NOVN.S,NOVARTIS,CH,CHF,Six Swiss Exchange,Large,5


In [191]:
display(df_sxxp.describe())
display(df_sxxp.describe(include="object"))

,Creation_Date,Rank (FINAL)
count,600.0,600.000000
mean,20260501.0,300.500000
std,0.0,173.349358
min,20260501.0,1.000000
25%,20260501.0,150.750000
50%,20260501.0,300.500000
75%,20260501.0,450.250000
max,20260501.0,600.000000


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_9318/3327613961.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df_sxxp.describe(include="object"))


,Internal_Key,ISIN,RIC,Instrument_Name,Country,Currency,Exchange,Index Membership
count,600,600,600,600,600,600,600,573
unique,600,600,600,600,17,8,16,3
top,546078,NL0010273215,ASML.AS,ASML HLDG,GB,EUR,London SE,Large
freq,1,1,1,1,128,297,127,200


In [192]:
# STOXX 600 spans EUR, UK, and a few ROW firms (Norway, Switzerland) — query all three regions
# and deduplicate, since a company will only appear in one region table
def _fetch_firm():
    regions = ["eur", "uk", "row"]
    parts = []
    for r in regions:
        try:
            df = db.raw_sql(f"""
                SELECT boardid, isin, boardname, hocountryname, sector,
                       mktcapitalisation, noemployees, revenue
                FROM boardex.{r}_wrds_company_profile
                WHERE isin IN {stoxx_isins_s}
            """)
            parts.append(df)
        except Exception as e:
            print(f"  [{r}] skipped: {e}")
    return pd.concat(parts, ignore_index=True).drop_duplicates("isin")

df_firm = load_or_fetch(DATA_DIR / "company_profile.csv", _fetch_firm, "Company profile")
df_firm.head()

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_9318/3472343370.py:10: DtypeWarning: Columns (0: hotelnumber, 1: hofaxnumber, 2: ccaddress1, 3: ccaddress2, 4: ccaddress3, 5: ccaddress4, 6: ccaddress5, 7: cccountryname, 8: cctelnumber, 9: ccfaxnumber, 10: ultimateparentcompanyid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cache_path)


[cache] Company profile: 445,004 rows


,isin,boardname,boardnameshort,hoaddress1,hoaddress2,hoaddress3,hoaddress4,hoaddress5,hocountryname,hotelnumber,...,successorcompanyid,ultimateparentcompanyid,boardid,ticker,countryofquote,primarystock,currency,mktcapitalisation,noemployees,revenue
0,NaN,1955 INVERSIONES SIMCAV SA,1955 INVERSIONES SIMCAV SA,NaN,NaN,NaN,NaN,NaN,Spain,NaN,...,NaN,NaN,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,FIRST ACTIVE PLC (De-listed 01/2004),FIRST ACTIVE PLC,NaN,NaN,NaN,NaN,NaN,Republic Of Ireland,NaN,...,783081.0,NaN,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,FIRST EAGLE FUND NV (De-listed 10/2011),FIRST EAGLE FUND NV,NaN,NaN,NaN,NaN,NaN,Netherlands Antilles,NaN,...,NaN,NaN,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2M INVEST A/S,2M INVEST A/S,NaN,NaN,NaN,NaN,NaN,Denmark,NaN,...,NaN,NaN,264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GB0004618236,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,THREADNEEDLE UK SELECT TRUST LTD,Ground Floor Dorey Court,Admiral Park,St Peter Port,NaN,GY1 2HT,Guernsey,+44 (0)1 481 727 111,...,NaN,NaN,296,UKT,ENGLAND AND WALES,Yes,NaN,NaN,NaN,NaN


In [193]:
display(df_firm.describe())
display(df_firm.describe(include="object"))

,companypolicy,cikcode,previouscompanyid,successorcompanyid,boardid,mktcapitalisation,noemployees,revenue
count,0.0,1.041000e+03,6.901000e+03,6.593000e+03,4.450040e+05,4770.000000,4.383000e+03,4489.00000
mean,NaN,1.426868e+06,2.082375e+06,2.493534e+06,2.397661e+06,6238.678826,1.278777e+04,4566.55714
std,NaN,3.652228e+05,1.226734e+06,1.085201e+06,1.074114e+06,29254.949981,6.514137e+04,17911.67812
min,NaN,1.970000e+03,1.000000e+01,5.540000e+02,8.000000e+00,0.000000,1.000000e+00,0.00000
25%,NaN,1.167379e+06,1.141672e+06,1.703653e+06,1.673212e+06,78.000000,1.585000e+02,46.00000
50%,NaN,1.475011e+06,2.213311e+06,2.652171e+06,2.567914e+06,308.500000,1.015000e+03,336.00000
75%,NaN,1.665584e+06,3.182669e+06,3.426112e+06,3.275926e+06,1771.500000,6.105000e+03,2116.00000
max,NaN,2.079106e+06,4.100544e+06,4.101470e+06,4.101635e+06,563017.000000,2.476748e+06,336131.00000


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_9318/2455531517.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df_firm.describe(include="object"))


,isin,boardname,boardnameshort,hoaddress1,hoaddress2,hoaddress3,hoaddress4,hoaddress5,hocountryname,hotelnumber,...,ccfaxnumber,sector,index,orgvisible,orgtype,ultimateparentcompanyid,ticker,countryofquote,primarystock,currency
count,6824,445004,445004,16905,12348,3593,14882,16462,445004,2698,...,2187,172966,1244,445004,445004,41559,6399,6823,6824,6408
unique,6677,444245,443604,14108,6404,1783,516,7563,55,2294,...,1907,52,89,2,10,10857,5914,50,2,1
top,LU1598757687,NB DISTRESSED DEBT INVESTMENT FUND LTD,NB DISTRESSED DEBT INVESTMENT FUND LTD,1 Royal Plaza,Amsterdam,Saint Peter Port,Paris,75008,Germany,+44 (0) 4 8175 0800,...,+46 (0) 8 735 57 44,Business Services,CDAX,No,Private,24776,BIO,FRANCE,Yes,USD
freq,4,12,12,28,243,139,1396,310,64496,12,...,7,21510,235,436587,389333,173,5,936,6230,6408


In [194]:
def _fetch_emp():
    # Chunk ISINs to stay well under SQL IN-clause limits
    chunk_size = 500
    parts = []
    isin_list = df_sxxp["ISIN"].dropna().str.strip().str.upper().unique().tolist()
    for i in range(0, len(isin_list), chunk_size):
        chunk = tuple(isin_list[i:i + chunk_size])
        parts.append(db.raw_sql("""
            SELECT isin, directorid, rolename, datestartrole, dateendrole
            FROM boardex.eur_wrds_dir_profile_emp
            WHERE isin IN %(isins)s
        """, params={"isins": chunk}))
    return pd.concat(parts, ignore_index=True)

df_emp = load_or_fetch(DATA_DIR / "executives_employment.csv", _fetch_emp, "Employment")
df_emp.head()

[cache] Employment: 95,197 rows


,isin,datestartrole,directorname,companyname,dateendrole,brdposition,rolename,fulltextdescription,ned,primarykeyid,directorid,companyid,hocountryname
0,GB00B1YW4409,1900-01-01,John Yetman,3I GROUP PLC,9999-12-31,No,Executive,Investment Executive,No,1654504,8428,294,United Kingdom - England
1,GB00B1YW4409,2005-07-06,Rod Perry,3I GROUP PLC,9999-12-31,No,Consultant,Also Head of International Advisory Board of V...,No,2003633,11777,294,United Kingdom - England
2,GB00B1YW4409,1900-01-01,John De Zulueta Greenebaum,3I GROUP PLC,9999-12-31,No,Advisor,NaN,No,2542008,6474,294,United Kingdom - England
3,GB00B1YW4409,2007-09-01,Peter Chambré,3I GROUP PLC,2015-12-28,No,Senior Advisor,NaN,No,3034743,26909,294,United Kingdom - England
4,GB00B1YW4409,2005-08-01,Vincent Guillaumot,3I GROUP PLC,2011-12-28,No,Associate Director,NaN,No,8354138,1669563,294,United Kingdom - England


In [195]:
display(df_emp.describe())
display(df_emp.describe(include="object"))

,primarykeyid,directorid,companyid
count,9.519700e+04,9.519700e+04,9.519700e+04
mean,1.086665e+07,1.605869e+06,3.494934e+05
std,4.892264e+06,8.766853e+05,7.973995e+05
min,2.190000e+02,3.600000e+01,2.940000e+02
25%,6.600024e+06,9.880770e+05,1.062700e+04
50%,1.142107e+07,1.600838e+06,2.435000e+04
75%,1.515448e+07,2.354550e+06,3.284700e+04
max,1.859297e+07,3.308284e+06,4.052924e+06


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_9318/2749970987.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df_emp.describe(include="object"))


,isin,datestartrole,directorname,companyname,dateendrole,brdposition,rolename,fulltextdescription,ned,hocountryname
count,95197,95197,95197,95197,95197,95197,95197,55987,95197,95197
unique,569,5885,50631,587,4073,2,9222,44480,2,26
top,DE0005140008,1900-01-01,Doctor Roland Busch,DEUTSCHE BANK AG,9000-01-01,No,Executive,Also Member of the Executive Committee,No,France
freq,1481,8931,19,1481,19320,74617,5439,754,78780,17224


In [196]:
# Director details available via WRDS SQL — query eur + uk + row regions
stoxx_directorids = tuple(df_emp["directorid"].dropna().unique().tolist())

def _fetch_exec():
    regions = ["eur", "uk", "row"]
    parts = []
    for r in regions:
        try:
            df = db.raw_sql(f"""
                SELECT directorid, age, gender
                FROM boardex.{r}_dir_profile_details
                WHERE directorid IN {stoxx_directorids}
            """)
            parts.append(df)
        except Exception as e:
            print(f"  [{r}] skipped: {e}")
    return pd.concat(parts, ignore_index=True).drop_duplicates("directorid")

df_exec = load_or_fetch(DATA_DIR / "executives_profile.csv", _fetch_exec, "Director details")
df_exec.head()

[cache] Director details: 332,924 rows


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_9318/3472343370.py:10: DtypeWarning: Columns (0: recreations) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cache_path)


,directorid,directorname,title,forename1,forename2,forename3,forename4,usualname,surname,suffixtitle,...,dod,age,gender,recreations,directorvisible,dobflag,dodflag,wealthxid,primaryroleid,networksize
0,16,David Shaw,Mr,David,Evans,NaN,NaN,NaN,Shaw,MBA,...,9999-12-31,74.0,M,NaN,Yes,10,55,112260.0,1083314.0,5675.0
1,36,Kevin Kelly,Mr,Kevin,John,NaN,NaN,NaN,Kelly,FCA FCIB,...,2012-01-04,70.0,M,NaN,Yes,10,10,2247015.0,7117.0,NaN
2,37,Klaus Zwickel,Mr,Klaus,NaN,NaN,NaN,NaN,Zwickel,NaN,...,9999-12-31,86.0,M,NaN,Yes,10,55,NaN,3.0,468.0
3,39,Doctor Christopher Albrecht,Doctor,Christopher,J,C,NaN,NaN,Albrecht,PhD,...,9999-12-31,87.0,M,NaN,Yes,10,55,NaN,50962.0,78.0
4,42,Umberto Agnelli,Mr,Umberto,NaN,NaN,NaN,NaN,Agnelli,NaN,...,2004-05-27,69.0,M,NaN,Yes,10,10,NaN,133504.0,NaN


In [197]:
display(df_exec.describe())
display(df_exec.describe(include="object"))

,directorid,age,dobflag,dodflag,wealthxid,primaryroleid,networksize
count,3.329240e+05,153778.000000,332924.000000,332924.000000,4.518500e+04,3.329170e+05,324465.000000
mean,1.903832e+06,60.289398,52.906579,54.603078,1.651513e+06,1.278838e+07,391.311386
std,8.771578e+05,11.435987,25.866251,4.164111,1.582051e+06,4.733453e+06,872.428029
min,1.600000e+01,19.000000,10.000000,10.000000,2.500000e+01,3.000000e+00,1.000000
25%,1.291695e+06,53.000000,30.000000,55.000000,3.017540e+05,9.928896e+06,32.000000
50%,2.011076e+06,60.000000,75.000000,55.000000,8.327250e+05,1.403005e+07,106.000000
75%,2.613906e+06,67.000000,75.000000,55.000000,2.635388e+06,1.669100e+07,372.000000
max,3.311347e+06,108.000000,75.000000,55.000000,5.063453e+06,1.859393e+07,22414.000000


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_9318/365339123.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df_exec.describe(include="object"))


,directorname,title,forename1,forename2,forename3,forename4,usualname,surname,suffixtitle,dob,dod,gender,recreations,directorvisible
count,332924,332924,332905,79642,10632,1198,14956,332916,16396,332924,332924,332919,21,332924
unique,323089,100,27608,18192,4127,635,3272,169463,1730,17045,2390,2,21,1
top,Lars Jensen,Mr,Michael,A,M,M,Chris,Müller,MBA,1900-01-01,9999-12-31,M,Languages: German English French,Yes
freq,15,235352,3791,1830,346,115,631,436,3641,179147,329907,261093,1,332924


In [198]:
# Filter by boardid (all executives, not just CEOs) — needed for pay dispersion analysis
stoxx_boardids = tuple(df_firm["boardid"].dropna().astype(int).unique().tolist())

def _fetch_renum():
    chunk_size = 500
    parts = []
    boardid_list = list(stoxx_boardids)
    for i in range(0, len(boardid_list), chunk_size):
        chunk = tuple(boardid_list[i:i + chunk_size])
        parts.append(db.raw_sql("""
            SELECT boardid, directorid, annualreportdate, rowtype,
                   salary, bonus, other, totalcompensation, currency, rolename
            FROM boardex.eur_dir_standard_remun
            WHERE boardid IN %(boardids)s
        """, params={"boardids": chunk}))
    return pd.concat(parts, ignore_index=True)

df_renum = load_or_fetch(DATA_DIR / "annual_renumeration.csv", _fetch_renum, "Annual remuneration")
df_renum.head()

[cache] Annual remuneration: 543,033 rows


,boardid,annualreportdate,rowtype,boardname,ned,directorname,rolename,currency,rolestatus,remchgelast,...,ltipvalue,intrvaloptaward,estvaloptaward,toteqatrisk,totremperiod,bonusratio,eqlinkremratio,wealthdelta,totaldirectcomp,perftotal
0,296,2016-12-01,SD Average,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,Yes,NaN,NaN,USD,NaN,0.14,...,NaN,NaN,NaN,NaN,35.0,NaN,NaN,2.0,35.0,NaN
1,296,2017-06-01,SD Average,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,Yes,NaN,NaN,USD,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,296,2017-06-01,Board Average,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,No,NaN,NaN,USD,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,296,2016-12-01,Board Average,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,No,NaN,NaN,USD,NaN,0.14,...,NaN,NaN,NaN,NaN,35.0,NaN,NaN,2.0,35.0,NaN
4,296,2017-06-01,SD Total,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,Yes,NaN,NaN,USD,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [199]:
display(df_renum.describe())
display(df_renum.describe(include="object"))

,boardid,remchgelast,recordid,directorid,salary,bonus,other,penempcon,totalcompensation,valtoteqheld,...,ltipvalue,intrvaloptaward,estvaloptaward,toteqatrisk,totremperiod,bonusratio,eqlinkremratio,wealthdelta,totaldirectcomp,perftotal
count,5.430330e+05,3.542200e+04,543033.000000,3.484050e+05,53602.000000,14243.000000,22001.000000,7964.000000,59027.000000,3.546500e+04,...,1.235000e+04,1156.000000,2008.000000,1.334000e+04,5.276500e+04,10098.000000,9969.000000,30229.000000,59027.000000,9213.000000
mean,1.127980e+06,4.825016e+02,0.861106,1.407068e+06,577.408847,1232.284561,319.803600,370.240834,794.434310,9.503040e+04,...,4.576520e+03,1516.256055,3654.532869,4.736168e+03,2.030425e+03,0.393986,0.547426,486.097952,955.229946,0.513926
std,1.172273e+06,2.877504e+04,1.335344,7.997451e+05,1390.301592,2116.915982,1577.749195,1348.091733,2253.513995,2.225096e+06,...,2.059874e+04,2517.353935,5742.875749,2.016820e+04,1.098199e+04,0.237404,0.231015,11332.775185,2907.784475,0.215058
min,2.960000e+02,-1.000000e+00,0.000000,3.100000e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,...,1.000000e+00,1.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.233800e+04,-2.000000e-02,0.000000,6.542510e+05,70.000000,72.000000,15.000000,24.000000,54.000000,7.600000e+01,...,4.050000e+02,86.000000,169.000000,3.057500e+02,7.800000e+01,0.230000,0.400000,1.000000,56.000000,0.380000
50%,8.360960e+05,6.000000e-02,0.000000,1.404226e+06,148.000000,581.000000,43.000000,83.000000,129.000000,3.980000e+02,...,1.526500e+03,442.500000,1334.000000,1.420000e+03,1.800000e+02,0.450000,0.540000,4.000000,142.000000,0.510000
75%,2.062329e+06,2.700000e-01,1.000000,2.016221e+06,502.000000,1495.000000,157.000000,285.000000,482.000000,3.198000e+03,...,3.883750e+03,1704.000000,4833.000000,3.919000e+03,8.940000e+02,0.570000,0.700000,30.000000,541.000000,0.650000
max,4.088842e+06,2.278240e+06,7.000000,3.308891e+06,40307.000000,37676.000000,63409.000000,26441.000000,55397.000000,1.462598e+08,...,1.005326e+06,22739.000000,62767.000000,1.005326e+06,1.018143e+06,1.000000,1.000000,793382.000000,82395.000000,1.000000


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_9318/7612780.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df_renum.describe(include="object"))


,annualreportdate,rowtype,boardname,ned,directorname,rolename,currency,rolestatus
count,543033,543033,543033,543033,513459,513459,543033,513459
unique,121,10,6994,2,73239,3454,1,161148
top,2022-12-01,Board Member,Helaba Landesbank Hessen-Thüringen,Yes,n.a.,n.a.,USD,n.a.
freq,59474,347384,416,344802,165054,165054,543033,165054


In [200]:
# Compustat Global fundamentals — cache-first, WRDS on miss
def _fetch_compustat():
    df_sec = db.raw_sql(f"""
        SELECT gvkey, TRIM(isin) AS isin
        FROM comp_global_daily.g_security
        WHERE TRIM(isin) IN {stoxx_isins_s}
    """)
    gvkeys = tuple(df_sec["gvkey"].unique().tolist())
    df_raw = db.raw_sql(f"""
        SELECT gvkey, fyear, at, sale, ebit, dltt, nicon, teq
        FROM comp_global_daily.g_funda
        WHERE gvkey IN {gvkeys}
          AND indfmt  = 'INDL'
          AND datafmt = 'HIST_STD'
          AND consol  = 'C'
          AND popsrc  = 'I'
          AND fyear BETWEEN 2000 AND 2024
    """)
    return df_raw.merge(
        df_sec[["gvkey", "isin"]].drop_duplicates("gvkey"),
        on="gvkey", how="left"
    )

df_cs = load_or_fetch(
    DATA_DIR / "fundamentals_annual.csv",
    _fetch_compustat,
    "Compustat Global fundamentals"
)
print(f"Unique gvkeys: {df_cs['gvkey'].nunique()}")
df_cs.head()

[cache] Compustat Global fundamentals: 9,409 rows
Unique gvkeys: 420


,gvkey,fyear,at,sale,ebit,dltt,nicon,teq,isin
0,1166,2000,777.940,935.212,191.772,31.660,94.272,NaN,NL0000334118
1,1166,2001,757.065,561.064,24.251,142.448,6.098,NaN,NL0000334118
2,1166,2002,653.841,518.802,-6.002,117.840,-29.862,NaN,NL0000334118
3,1166,2003,672.301,581.868,14.199,179.123,-29.320,NaN,NL0000334118
4,1166,2004,823.834,754.245,88.441,193.345,24.039,NaN,NL0000334118


In [201]:
display(df_cs.describe())
display(df_cs.describe(include="object"))

,gvkey,fyear,at,sale,ebit,dltt,nicon,teq
count,9409.000000,9409.000000,9.397000e+03,9.397000e+03,9.392000e+03,9.397000e+03,8.592000e+03,5994.000000
mean,160578.059943,2012.637156,5.462313e+04,3.119816e+04,3.798558e+03,9.704964e+03,1.814528e+03,12942.110620
std,94085.761851,7.165020,1.216917e+06,5.264751e+05,9.659941e+04,1.829217e+05,4.585916e+04,25122.410554
min,1166.000000,2000.000000,0.000000e+00,0.000000e+00,-3.249900e+04,0.000000e+00,-3.750000e+04,-6016.000000
25%,100760.000000,2007.000000,2.807254e+03,2.050773e+03,1.920000e+02,2.718240e+02,9.660000e+01,1391.450000
50%,104761.000000,2013.000000,8.449749e+03,6.219000e+03,5.748500e+02,1.556000e+03,3.280000e+02,3747.577500
75%,234117.000000,2019.000000,2.921600e+04,1.996000e+04,1.945750e+03,5.675708e+03,1.139293e+03,12295.750000
max,370750.000000,2024.000000,1.003320e+08,4.761300e+07,9.204000e+06,1.656900e+07,4.236000e+06,381200.000000


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_9318/681328450.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df_cs.describe(include="object"))


,isin
count,9409
unique,420
top,GB0006731235
freq,28


---
## Step 1: Data Merging & CEO Turnover Events

In [202]:
# --- 1.1  Restrict to STOXX 600 universe ---
stoxx600_isins = set(df_sxxp["ISIN"])
df_firm_sxxp = (
    df_firm[df_firm["isin"].isin(stoxx600_isins)]
    .drop_duplicates(subset=["isin"])
    .copy()
)
stoxx600_boardids = set(df_firm_sxxp["boardid"])

print(f"STOXX 600 ISINs in index:           {len(stoxx600_isins)}")
print(f"Matched firms in BoardEx:           {len(df_firm_sxxp)}")
print(f"Unique board IDs in universe:       {len(stoxx600_boardids)}")
df_firm_sxxp[["isin", "boardid", "boardname", "hocountryname", "sector", "mktcapitalisation"]].head()

STOXX 600 ISINs in index:           600
Matched firms in BoardEx:           466
Unique board IDs in universe:       466


,isin,boardid,boardname,hocountryname,sector,mktcapitalisation
29,NL0000852564,384,AALBERTS NV (Aalberts Industries NV prior to 0...,Netherlands,Engineering & Machinery,3978.0
45,CH0012221716,422,ABB LTD,Switzerland,Engineering & Machinery,182588.0
74,ES0125220311,595,ACCIONA SA,Spain,Renewable Energy,12044.0
75,FR0000120404,598,ACCOR SA,France,Leisure & Hotels,11568.0
83,ES0132105018,626,ACERINOX SA,Spain,Steel & Other Metals,3929.0


In [203]:
# --- 1.2  Identify CEO roles within STOXX 600 firms ---
CEO_PATTERN = r"chief executive|(?<!\w)ceo(?!\w)|managing director"
df_emp["is_ceo"] = df_emp["rolename"].str.lower().str.contains(CEO_PATTERN, na=False, regex=True)
df_ceo_emp = df_emp[df_emp["is_ceo"] & df_emp["isin"].isin(stoxx600_isins)].copy()

print(f"CEO role-records in STOXX 600 firms: {len(df_ceo_emp)}")
df_ceo_emp["rolename"].value_counts().head(10)

CEO role-records in STOXX 600 firms: 6653


rolename
Division CEO                       2390
CEO                                 529
Regional CEO                        483
President/CEO                       275
Chairman/CEO                        268
Division Chairman/Division CEO      231
Division President/Division CEO     153
Deputy CEO                          137
Division President/CEO              133
Division Deputy CEO                 112
Name: count, dtype: int64

In [204]:
# --- 1.3  Parse dates and define CEO turnover events ---
df_ceo_emp["start"] = pd.to_datetime(df_ceo_emp["datestartrole"], errors="coerce")
df_ceo_emp["end"]   = pd.to_datetime(df_ceo_emp["dateendrole"],   errors="coerce")

UNKNOWN_START = pd.Timestamp("1900-01-01")

# Drop records with no real start date
df_ceo_dated = df_ceo_emp[df_ceo_emp["start"] > UNKNOWN_START].copy()
df_ceo_dated["start_year"] = df_ceo_dated["start"].dt.year

# Deduplicate to one record per (isin, directorid, start_year)
df_ceo_starts = (
    df_ceo_dated
    .sort_values(["isin", "start_year"])
    .drop_duplicates(subset=["isin", "directorid", "start_year"])
    .copy()
)

# CEO sequence per firm; turnover = new CEO arriving (rank > 1 excludes initial incumbent)
df_ceo_starts["ceo_seq"] = (
    df_ceo_starts.groupby("isin")["start_year"].rank(method="first").astype(int)
)
df_turnover = (
    df_ceo_starts[df_ceo_starts["ceo_seq"] > 1][["isin", "directorid", "start_year"]]
    .rename(columns={"start_year": "year"})
    .assign(ceo_turnover=1)
)

print(f"CEO turnover events identified: {len(df_turnover)}")
df_turnover.head()

CEO turnover events identified: 5806


,isin,directorid,year,ceo_turnover
80018,AT0000606306,334105,2001,1
80176,AT0000606306,482492,2010,1
80034,AT0000606306,482228,2011,1
80043,AT0000606306,801312,2013,1
80177,AT0000606306,482492,2013,1


In [205]:
# --- 1.4  Build firm-year CEO panel ---
OPEN_END_DATES = {pd.Timestamp("9999-12-31"), pd.Timestamp("9000-01-01")}
YEAR_MIN, YEAR_MAX = 2000, 2024

df_ceo_dated["end_year"] = df_ceo_dated["end"].apply(
    lambda x: YEAR_MAX if pd.isna(x) or x in OPEN_END_DATES else int(x.year)
)

rows = []
for _, row in df_ceo_dated.iterrows():
    y_start = max(row["start_year"], YEAR_MIN)
    y_end   = min(row["end_year"],   YEAR_MAX)
    for y in range(y_start, y_end + 1):
        rows.append({"isin": row["isin"], "year": y, "directorid": row["directorid"]})

# One CEO per firm-year (keep first by start date, already sorted)
df_ceo_panel = pd.DataFrame(rows).drop_duplicates(subset=["isin", "year"])

print(f"Firm-year CEO observations: {len(df_ceo_panel):,}")
df_ceo_panel.head()

Firm-year CEO observations: 8,517


,isin,year,directorid
0,GB00B1YW4409,2008,1297838
1,GB00B1YW4409,2009,1297838
2,GB00B1YW4409,2010,1297838
3,GB00B1YW4409,2011,1297838
4,GB00B1YW4409,2012,1297838


In [206]:
# --- 1.5  Merge remuneration, firm attributes, and executive demographics ---
df_renum["year"] = pd.to_datetime(df_renum["annualreportdate"], errors="coerce").dt.year

# Individual compensation rows only (exclude aggregate summary rows)
df_renum_indiv = df_renum[
    df_renum["boardid"].isin(stoxx600_boardids) &
    df_renum["directorid"].notna() &
    ~df_renum["rowtype"].str.lower().str.contains("average|total", na=False)
].sort_values("totalcompensation", ascending=False).drop_duplicates(["boardid", "directorid", "year"])

FIRM_COLS  = ["isin", "boardid", "boardname", "hocountryname", "sector",
              "mktcapitalisation", "noemployees", "revenue"]
RENUM_COLS = ["boardid", "directorid", "year",
              "salary", "bonus", "other", "totalcompensation", "currency", "rolename"]
EXEC_COLS  = ["directorid", "age", "gender"]

df_panel = (
    df_ceo_panel
    .merge(df_firm_sxxp[FIRM_COLS].drop_duplicates("isin"), on="isin", how="left")
    .merge(df_renum_indiv[RENUM_COLS], on=["boardid", "directorid", "year"], how="left")
    .merge(df_exec[EXEC_COLS].drop_duplicates("directorid"), on="directorid", how="left")
    .merge(df_turnover[["isin", "year", "ceo_turnover"]], on=["isin", "year"], how="left")
)
df_panel["ceo_turnover"] = df_panel["ceo_turnover"].fillna(0).astype(int)

print(f"Panel shape:               {df_panel.shape}")
print(f"Turnover events in panel:  {df_panel['ceo_turnover'].sum()}")
print(f"Compensation coverage:     {df_panel['totalcompensation'].notna().sum():,} / {len(df_panel):,} rows")
df_panel.head()

Panel shape:               (11146, 19)
Turnover events in panel:  5400
Compensation coverage:     711 / 11,146 rows


,isin,year,directorid,boardid,boardname,hocountryname,sector,mktcapitalisation,noemployees,revenue,salary,bonus,other,totalcompensation,currency,rolename,age,gender,ceo_turnover
0,GB00B1YW4409,2008,1297838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.0,F,0
1,GB00B1YW4409,2009,1297838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.0,F,0
2,GB00B1YW4409,2010,1297838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.0,F,0
3,GB00B1YW4409,2011,1297838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.0,F,0
4,GB00B1YW4409,2012,1297838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.0,F,1


In [207]:
# --- 1.6  Merge Compustat fundamentals ---
# df_cs is already loaded from cache/WRDS in the data-loading section above
df_cs = df_cs.rename(columns={"fyear": "year"})

# Derive financial ratios needed as covariates
df_cs["roa"]         = df_cs["nicon"] / df_cs["at"]
df_cs["ebit_margin"] = df_cs["ebit"]  / df_cs["sale"]
df_cs["leverage"]    = df_cs["dltt"]  / df_cs["at"]
df_cs["log_assets"]  = np.log(df_cs["at"].replace(0, np.nan))

CS_COLS = ["isin", "year", "roa", "ebit_margin", "leverage", "log_assets", "at", "sale"]
df_panel = df_panel.merge(
    df_cs[CS_COLS].drop_duplicates(["isin", "year"]),
    on=["isin", "year"], how="left"
)

print(f"Panel shape after Compustat merge: {df_panel.shape}")
print(f"ROA coverage:       {df_panel['roa'].notna().sum():,} / {len(df_panel):,} rows")
print(f"Leverage coverage:  {df_panel['leverage'].notna().sum():,} / {len(df_panel):,} rows")

df_panel.to_csv(DATA_DIR / "panel_ceo_merged.csv", index=False)
print(f"\nSaved → panel_ceo_merged.csv  [{df_panel.shape[0]:,} rows × {df_panel.shape[1]} cols]")
df_panel[["isin", "year", "boardname", "directorid", "totalcompensation",
          "ceo_turnover", "roa", "leverage", "log_assets"]].head(10)

Panel shape after Compustat merge: (11146, 25)
ROA coverage:       7,806 / 11,146 rows
Leverage coverage:  8,000 / 11,146 rows

Saved → panel_ceo_merged.csv  [11,146 rows × 25 cols]


,isin,year,boardname,directorid,totalcompensation,ceo_turnover,roa,leverage,log_assets
0,GB00B1YW4409,2008,NaN,1297838,NaN,0,NaN,NaN,NaN
1,GB00B1YW4409,2009,NaN,1297838,NaN,0,NaN,NaN,NaN
2,GB00B1YW4409,2010,NaN,1297838,NaN,0,NaN,NaN,NaN
3,GB00B1YW4409,2011,NaN,1297838,NaN,0,NaN,NaN,NaN
4,GB00B1YW4409,2012,NaN,1297838,NaN,1,NaN,NaN,NaN
5,GB00B1YW4409,2013,NaN,1297838,NaN,0,NaN,NaN,NaN
6,GB00B1YW4409,2014,NaN,1297838,NaN,0,NaN,NaN,NaN
7,GB00B1YW4409,2015,NaN,342366,NaN,0,NaN,NaN,NaN
8,GB00B1YW4409,2016,NaN,342366,NaN,0,NaN,NaN,NaN
9,GB00B1YW4409,2017,NaN,342366,NaN,0,NaN,NaN,NaN


---
## Step 2: Data Cleaning

### 2.1 CPI Deflation (country-specific, base 2015 = 100)

In [208]:
import requests, time

# --- 2.1  Map firm HQ country to ISO-2 ---
COUNTRY_ISO = {
    "Austria": "AT", "Belgium": "BE", "Denmark": "DK", "Finland": "FI",
    "France": "FR", "Germany": "DE", "Ireland": "IE", "Republic Of Ireland": "IE",
    "Italy": "IT", "Luxembourg": "LU", "Netherlands": "NL", "Norway": "NO",
    "Poland": "PL", "Portugal": "PT", "Spain": "ES", "Sweden": "SE",
    "Switzerland": "CH", "United Kingdom": "GB",
    "United Kingdom - England": "GB", "United Kingdom - Scotland": "GB",
    "United Kingdom - Wales": "GB", "United Kingdom - Northern Ireland": "GB",
    # Crown dependencies → UK prices
    "Isle Of Man": "GB", "Jersey": "GB",
    # US-HQ outliers in BoardEx
    "United States": "US",
}

df_panel["iso2"] = df_panel["hocountryname"].map(COUNTRY_ISO)
unmapped = df_panel[df_panel["iso2"].isna() & df_panel["hocountryname"].notna()]["hocountryname"].unique()
if len(unmapped):
    print(f"Unmapped countries: {unmapped}")
else:
    print("All countries mapped successfully")

# --- 2.2  Fetch CPI from World Bank (cached to avoid repeated API calls) ---
CPI_CACHE = DATA_DIR / "cpi_country_annual.csv"

if CPI_CACHE.exists():
    df_cpi = pd.read_csv(CPI_CACHE)
    print(f"Loaded CPI from cache ({len(df_cpi)} records)")
else:
    iso_codes = sorted(df_panel["iso2"].dropna().unique().tolist())
    records = []
    for iso in iso_codes:
        url = (
            f"https://api.worldbank.org/v2/country/{iso}/indicator/FP.CPI.TOTL"
            f"?date=2000:2024&format=json&per_page=100"
        )
        for attempt in range(3):
            try:
                raw = requests.get(url, timeout=60).json()[1]
                records.extend([
                    {"iso2": d["country"]["id"], "year": int(d["date"]), "cpi": d["value"]}
                    for d in raw if d["value"] is not None
                ])
                break
            except Exception as e:
                print(f"  {iso} attempt {attempt+1} failed: {e}")
                time.sleep(3)
    df_cpi = pd.DataFrame(records)
    df_cpi.to_csv(CPI_CACHE, index=False)
    print(f"Fetched and cached CPI ({len(df_cpi)} records, {df_cpi['iso2'].nunique()} countries)")

# --- 2.3  Rebase to 2015 = 100 ---
cpi_2015 = (
    df_cpi[df_cpi["year"] == 2015]
    .set_index("iso2")["cpi"]
    .rename("cpi_2015")
)
df_cpi = df_cpi.join(cpi_2015, on="iso2")
df_cpi["cpi_idx"] = df_cpi["cpi"] / df_cpi["cpi_2015"] * 100

# --- 2.4  Merge and deflate ---
df_panel = df_panel.merge(
    df_cpi[["iso2", "year", "cpi_idx"]],
    on=["iso2", "year"], how="left"
)

# Compensation (levels only — ratios like ROA/leverage are already real)
for col in ["totalcompensation", "salary", "bonus", "other"]:
    df_panel[f"real_{col}"] = df_panel[col] / df_panel["cpi_idx"] * 100

# Firm fundamentals
df_panel["real_at"]         = df_panel["at"]   / df_panel["cpi_idx"] * 100
df_panel["real_sale"]       = df_panel["sale"] / df_panel["cpi_idx"] * 100
df_panel["real_log_assets"] = np.log(df_panel["real_at"].replace(0, np.nan))

print(f"CPI coverage:  {df_panel['cpi_idx'].notna().sum():,} / {len(df_panel):,} panel rows")
print(f"Real pay rows: {df_panel['real_totalcompensation'].notna().sum():,}")
df_panel[["isin", "year", "hocountryname", "iso2", "cpi_idx",
          "totalcompensation", "real_totalcompensation"]].dropna(subset=["totalcompensation"]).head(8)

All countries mapped successfully
Loaded CPI from cache (450 records)
CPI coverage:  10,234 / 11,146 panel rows
Real pay rows: 711


,isin,year,hocountryname,iso2,cpi_idx,totalcompensation,real_totalcompensation
176,DE000A1EWWW0,2016,Germany,DE,100.491747,1536.0,1528.483727
177,DE000A1EWWW0,2017,Germany,DE,102.008665,4458.0,4370.216992
178,DE000A1EWWW0,2018,Germany,DE,103.775627,3945.0,3801.470647
179,DE000A1EWWW0,2019,Germany,DE,105.275870,3846.0,3653.258831
180,DE000A1EWWW0,2020,Germany,DE,105.428391,4191.0,3975.210055
181,DE000A1EWWW0,2021,Germany,DE,108.661528,4345.0,3998.655335
192,DE000A1EWWW0,2023,Germany,DE,123.034932,4451.0,3617.671759
193,DE000A1EWWW0,2024,Germany,DE,125.811213,4556.0,3621.298835


### 2.2 Missing-value Analysis

In [ ]:
# --- 2.2  Missing-value summary ---
pay_cols  = ["totalcompensation", "salary", "bonus", "other",
             "real_totalcompensation", "real_salary", "real_bonus", "real_other"]
firm_cols = ["roa", "ebit_margin", "leverage", "real_log_assets"]

miss = pd.DataFrame({
    "non-null":   df_panel[pay_cols + firm_cols].notna().sum(),
    "missing":    df_panel[pay_cols + firm_cols].isna().sum(),
    "coverage %": (df_panel[pay_cols + firm_cols].notna().mean() * 100).round(1)
})
print(f"Panel rows: {len(df_panel):,}")
display(miss)

# Compensation coverage by year
pay_by_year = (
    df_panel.groupby("year")["totalcompensation"]
    .agg(n_obs="count", n_total="size")
    .assign(pct=lambda x: (x["n_obs"] / x["n_total"] * 100).round(1))
)
print("\nCompensation coverage by year:")
display(pay_by_year[pay_by_year["n_obs"] > 0])

# Compensation coverage by country
pay_by_country = (
    df_panel.groupby("hocountryname")["totalcompensation"]
    .agg(n_obs="count", n_total="size")
    .assign(pct=lambda x: (x["n_obs"] / x["n_total"] * 100).round(1))
    .sort_values("pct", ascending=False)
)
print("\nCompensation coverage by country:")
display(pay_by_country)

### 2.3 Winsorization of Pay (p1 / p99)

In [ ]:
# --- 2.3  Winsorize real compensation at p1/p99 (among non-null rows) ---
def winsorize_series(s, lo=0.01, hi=0.99):
    """Clip to [p1, p99]; NaN-safe."""
    q_lo, q_hi = s.quantile([lo, hi])
    return s.clip(lower=q_lo, upper=q_hi)

real_pay_cols = ["real_totalcompensation", "real_salary", "real_bonus", "real_other"]
for col in real_pay_cols:
    df_panel[f"w_{col}"] = winsorize_series(df_panel[col])

# Log of winsorized total pay (used as dependent variable in regressions)
df_panel["log_w_real_pay"] = np.log(df_panel["w_real_totalcompensation"].replace(0, np.nan))

# Quick sanity check
stats = df_panel[["real_totalcompensation", "w_real_totalcompensation", "log_w_real_pay"]].describe()
display(stats)

# How many rows clipped at each tail?
p1 = df_panel["real_totalcompensation"].quantile(0.01)
p99 = df_panel["real_totalcompensation"].quantile(0.99)
n_lo = (df_panel["real_totalcompensation"] < p1).sum()
n_hi = (df_panel["real_totalcompensation"] > p99).sum()
print(f"
p1={p1:,.0f}, p99={p99:,.0f}")
print(f"Rows clipped at lower tail: {n_lo}")
print(f"Rows clipped at upper tail: {n_hi}")

### 2.4 Save Clean Panel

In [ ]:
# --- 2.4  Persist the clean panel ---
PANEL_CLEAN = DATA_DIR / "panel_clean.csv"
df_panel.to_csv(PANEL_CLEAN, index=False)
print(f"Saved: {PANEL_CLEAN}")
print(f"Shape: {df_panel.shape}")
print(f"Columns: {df_panel.columns.tolist()}")

## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference
- [X] Causal graph / DAG (DoWhy)
- [X] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

*Justification:* We will model CEO turnover as a governance shock in a DAG and use backdoor adjustment to control for firm fundamentals that affect both turnover and pay. We will estimate the causal effect of turnover using pre/post (event‑study style) comparisons around the turnover year.

### 4b. Supervised Learning
- [X] Linear / Ridge / Lasso regression
- [X] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [X] Decision Tree / Random Forest
- [X] Gradient Boosting (XGBoost / LightGBM / sklearn GBM)
- [ ] Neural network (regression or classification)
- [ ] Other: ___

*Justification:* Supervised models will benchmark expected compensation conditional on firm and executive characteristics. Linear models provide interpretable baselines and covariate effects, while tree-based and boosting models capture nonlinearities/interactions for more accurate counterfactual pay predictions. We will use Optuna to tune boosting hyperparameters (e.g., depth, learning rate, subsampling) to avoid overfitting and compare against simpler baselines.

### 4c. Unsupervised Learning / Generative Models
- [X] K-Means clustering
- [ ] Hierarchical clustering
- [X] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification:* Clustering will segment firms into comparable peer groups before causal estimation and highlight heterogeneous effects across turnover regimes. A VAE will learn low-dimensional representations of firm/executive profiles to detect anomalous pay structures and support exploratory subgroup analysis.

## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

- RMSE


## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | Achmad | Data collection & merging from BoardEx/Compustat; define CEO turnover events |
| 2 | Kajetan | Data cleaning, missing-value strategy, currency/inflation adjustments |
| 3 | Achmad | EDA + descriptive stats; define pay dispersion metrics |
| 4 | Kajetan | Causal inference block (DAG, event-window design around turnover, ATE estimation) |
| 5 | Achmad | Supervised learning benchmark models + heterogeneity analysis |
| 6 | Kajetan | Unsupervised/generative clustering/embeddings; peer-group analysis |
| 7 | Achmad + Kajetan | Synthesis, robustness checks, and final write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
